# CropGuard — Potato Disease Detector (3 classes, 100 epochs)

SIH 2026 PS26131. Trains the detector the FastAPI backend serves.

**Scope: potato only, three classes** — `potato_early_blight`, `potato_late_blight`, `potato_healthy`.

Run on Kaggle with **GPU T4 x2** (Settings → Accelerator) and internet ON.
Add the dataset `abdallahalidev/plantvillage-dataset` via *Add Input*.

> **Kaggle, not Colab.** Colab disconnects idle sessions and reclaims GPUs mid-run,
> which kills a 100-epoch job with no checkpoint. Kaggle gives a 30 hr/week quota and
> a 12 hr session limit — comfortably more than this run needs.

The laptop never sees the dataset; only `ml/weights/` (a few MB) comes back down.

**Pipeline:** clone repo → build stratified balanced dataset → train → evaluate lab *vs* field
→ tune per-class thresholds → export + verify ONNX → benchmark → download.


In [ ]:
import subprocess

try:
    subprocess.run(["pip", "install", "-q", "wandb"], check=True)
    import wandb
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    wandb.login(key=secrets.get_secret("WANDB_API_KEY"))
    print("wandb ready")
except Exception as e:
    print(f"wandb skipped: {e}")
    print("Training will continue without W&B logging.")

## 1. Environment and repo


In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim
import torch, ultralytics
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__)


Clone the repo so the notebook uses the same scripts as the backend — rather than a
copy of the logic that can silently drift out of sync with `taxonomy.CLASS_NAMES`.


In [ ]:
import os, shutil, sys
from pathlib import Path

REPO = Path('/kaggle/working/Crop_detection_management')
REPO_URL = 'https://github.com/aliviahossain/Crop_detection_management.git'
BRANCH = 'main'  # integration branch; set a commit hash for a reproducible released artifact

if not REPO.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}
else:
    # /kaggle/working persists across reruns. Force the existing checkout to the
    # intended revision so a rerun never trains on stale code.
    !cd {REPO} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD

if REPO.exists():
    os.chdir(REPO)
    !git -C {REPO} log -1 --oneline
    print('repo ready:', REPO)
else:
    print('No repo — falling back to the inline cells below.')

## 2. Build the dataset — stratified and balanced

PlantVillage potato is roughly **1000 / 1000 / 152**, a 6.5:1 imbalance where the minority
class is `healthy` — the one that lets the system say *do not spray*. Left alone the model
under-predicts it and the system recommends chemicals it should not.

`prepare_dataset.py` handles this explicitly:

- **stratified splits** with exact per-class quotas (deterministic by content hash, so an
  image keeps its split across reruns and nothing leaks train→val),
- **minimum 20 val images per class**, because a metric off 15 images is noise,
- **`--cap-train`** caps majority classes in the *train split only* — never val/test,
- **`--oversample-min`** repeats minority training images up to the majority count,
- separate **`data_lab.yaml` / `data_field.yaml`** val lists for honest evaluation.


In [ ]:
!python ml/prepare_dataset.py \
    --plantvillage /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color \
    --out /kaggle/working/potato_yolo \
    --cap-train 400 --oversample-min --clean


### Add field-condition images — the highest-value step

A model trained on PlantVillage alone learns *leaf on uniform grey background* as much as
it learns disease. It will score ~0.95+ on its own test split and degrade badly on a real
phone photo with soil, shadow and overlapping leaves.

If you have a Roboflow export or the PlantDoc potato subset, add it as a Kaggle input and
re-run with `--annotated`. Real boxes take priority over the weak full-frame boxes generated
for PlantVillage. See `ml/DATASETS.md` for the dataset comparison.


In [ ]:
# Uncomment once you have a field-condition dataset attached as an input.
# --remap maps ITS class ids onto ours (0=early, 1=late, 2=healthy); unlisted ids are dropped.

# !python ml/prepare_dataset.py \
#     --plantvillage /kaggle/input/plantvillage-dataset/color \
#     --annotated    /kaggle/input/<your-field-dataset> \
#     --remap '0=0,1=1,2=2' \
#     --out /kaggle/working/potato_yolo \
#     --cap-train 400 --oversample-min --clean


In [ ]:
import json
manifest = json.loads(Path('/kaggle/working/potato_yolo/dataset_manifest.json').read_text())
print(json.dumps({k: v for k, v in manifest.items() if k != 'by_source'}, indent=2))

if manifest['field_val_images'] == 0:
    print('\n*** Every metric from this run will be a LAB metric. ***')
    print('*** Do not present it as field accuracy. See ml/DATASETS.md. ***')


## 3. Train — 100 epochs

**`yolov8s`, not `yolov8n`.** Nano is the least accurate variant in the family, and accuracy
is what matters when a wrong answer means the wrong pesticide. `s` roughly triples the
parameters (11.2M vs 3.2M) for a real mAP gain and still fits the CPU latency budget —
which step 7 measures rather than assumes.

Train `n` as well if you want the offline/on-device story: ship `s` on the server, `n` in a
phone build.

**Augmentation is deliberate, and matters more than usual on a dataset this small.**
Geometry is augmented freely; colour is barely touched. Lesion *colour* is the diagnostic
signal separating early from late blight — aggressive hue jitter teaches the model to ignore
the one feature that matters. `flipud=0` because leaves are photographed the right way up,
and `close_mosaic=10` runs the last 10 epochs without mosaic so validation resembles real
single-image inference.


In [ ]:
MODEL = 'yolov8s.pt'   # 'yolov8n.pt' for the mobile/offline variant
EPOCHS = 100
DATA = '/kaggle/working/potato_yolo/data.yaml'

!python ml/train_yolo.py \
    --data {DATA} \
    --model {MODEL} \
    --epochs {EPOCHS} \
    --imgsz 640 --batch 16 \
    --project /kaggle/working/runs \
    --export-dir /kaggle/working/ml_weights


## 4. Evaluate — lab vs field, reported separately

A single mAP number is the most misleading output this pipeline can produce. `evaluate.py`
reports the lab and field splits side by side and warns when the gap exceeds 0.20 mAP50 —
the signature of a model that learnt the background rather than the disease.


In [ ]:
!python ml/evaluate.py \
    --weights /kaggle/working/ml_weights/best.pt \
    --data {DATA} \
    --out /kaggle/working/ml_weights/evaluation.json


In [ ]:
from IPython.display import Image, display
run = sorted(Path('/kaggle/working/runs').glob('*'))[-1]
print('run:', run)
for plot in ('confusion_matrix_normalized.png', 'results.png',
             'PR_curve.png', 'val_batch0_pred.jpg'):
    p = run / plot
    if p.exists():
        print(plot)
        display(Image(filename=str(p), width=760))


Read the **normalised confusion matrix** before trusting any headline number. The row that
matters most is `potato_healthy`: anything it leaks into a disease class is an unnecessary
spray, and any disease leaking into `healthy` is a missed infection — the delayed-treatment
failure the problem statement is about.


## 5. Tune per-class confidence thresholds

The stock `conf=0.25` assumes a false positive and a false negative cost the same. They do
not:

- **Missing late blight** can destroy the field in 7–10 days.
- **A false positive** costs a spray — but the backend's triage layer already withholds the
  dose table below its low-confidence threshold and routes the case to an extension officer.
  There is a second line of defence against false positives and none against false negatives.
- **A false `healthy`** is the dangerous direction of that class.

So disease classes are tuned for **F2** (recall-weighted) and `potato_healthy` for **F0.5**
(precision-weighted). The backend loads the resulting `thresholds.json` automatically.


In [ ]:
!python ml/tune_thresholds.py \
    --weights /kaggle/working/ml_weights/best.pt \
    --data {DATA} \
    --out /kaggle/working/ml_weights/thresholds.json

print(json.dumps(json.loads(
    Path('/kaggle/working/ml_weights/thresholds.json').read_text()
)['per_class'], indent=2))


## 6. Export to ONNX and verify

ONNX Runtime on CPU is the serving path: no torch on the laptop, a few hundred ms per image,
and the same artifact an offline on-device build would ship. The verification step checks the
output tensor is the `(1, 4+nc, N)` shape the backend decoder expects with `nc == 3` — a wrong
export would otherwise surface only as bad predictions in the field.


In [ ]:
!python ml/export_onnx.py \
    --weights /kaggle/working/ml_weights/best.pt \
    --out /kaggle/working/ml_weights/best.onnx


## 7. Benchmark CPU inference

Settles the model-size trade with numbers instead of assumptions. If a judge asks about
inference speed, this is the answer — and it is measured.


In [ ]:
!python ml/benchmark_inference.py \
    --model /kaggle/working/ml_weights/best.onnx \
    --runs 30

# Kaggle CPUs are not your deployment hardware — re-run this locally on the
# machine that will actually serve requests before quoting a latency figure.


## 8. Download

Download `/kaggle/working/ml_weights/` from the Output panel and place its contents in
`ml/weights/` in the repo:

| File | Purpose |
|---|---|
| `best.onnx` | **the serving artifact** — the backend loads this |
| `best.pt` | source checkpoint, for re-export and further tuning |
| `thresholds.json` | per-class confidence thresholds, loaded automatically |
| `metrics.json` | training metrics |
| `evaluation.json` | lab vs field comparison and the honest headline number |

`GET /meta/health` stops reporting `detection_model_missing`, and `GET /detect/status`
shows the tuned per-class thresholds in use.

---

### Next increment

Once extension officers have validated field cases, run `ml/export_feedback.py` locally to
package them, upload that folder as a Kaggle input, and pass it as `--annotated` in step 2.
That is the retraining loop — and it is how the field split grows past zero.


In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/runs/yolov8s_3class_100e/weights/best.pt")
dst = Path("/kaggle/working/ml_weights")
dst.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst / "best.pt")
print("Saved:", dst / "best.pt")

In [ ]:
yaml_content = """path: /kaggle/working/potato_yolo
train: images/train
val: images/test
test: images/test

names:
  0: potato_early_blight
  1: potato_late_blight
  2: potato_healthy
"""
Path("/kaggle/working/potato_yolo/data_test.yaml").write_text(yaml_content)
print("data_test.yaml written")

In [ ]:
# Evaluate on test split
!python ml/evaluate.py \
    --weights /kaggle/working/ml_weights/best.pt \
    --data /kaggle/working/potato_yolo/data_test.yaml \
    --out /kaggle/working/ml_weights/evaluation_test.json